In [ ]:
%run ./extract.ipynb

new_data = extract_data()

print(len(new_data['results']))

In [ ]:
# Deduplication Function
def remove_duplicates(records, key_columns):
    unique_records = []
    seen_ids = set()

    for record in records:
        values = []

        for column in key_columns:
            values.append(record[column])

        key = tuple(values)
    
        if key not in seen_ids:
            unique_records.append(record)
            seen_ids.add(key)

    return unique_records

In [ ]:
# validation function 
def validate_records(records, required_columns):
    # create an empty list that will store the valid records (dictionaries)
    valid_records = []

    # examines one dict at a time
    for record in records:
        
    # assume that the record is valid
        valid = True

        # iterate over the columns un the required columns
        for column in required_columns:
            # if the value of the column of the dictionary is None, then set valid to False
            if record.get(column) is None:
                valid = False
            # If the value is false, interrupts the loop since i do not need to check more columns    
                break

        # if the valued is not None, and thus valid is True, then add it to the valid_records list
        if valid == True:
            valid_records.append(record)

    return valid_records

In [ ]:

# define new function for creating the dict
# paper

def paper_transform(paper_extracted):
   
#conditionals for missing data
   open_access = paper_extracted.get('open_access')
# conditional statement is_oa
   if open_access is not None:
      is_oa = open_access.get('is_oa')
   else:
      is_oa = None

# conditional statement source_id
   primary_location = paper_extracted.get('primary_location')
   if primary_location is not None:
      source = primary_location.get('source')
   else: 
      source = None   
      
   if source is not None:
      source_id = source.get('id')
   else:
      source_id = None
   
# Create the Dictionary
   paper = {'id': paper_extracted.get('id'),
         'title': paper_extracted.get('title'),
         'year': paper_extracted.get('publication_year'),
         'is_oa': is_oa,
         'type': paper_extracted.get('type'),
         'source_id': source_id        
         }
   return paper

paper = []
for publication in new_data['results']:
   paper.append(paper_transform(publication))
# print(len(paper)) prints how many papers in the data
# print(paper)   

#test

# test_papers = [
    
#     {
#         'id': 'W3',
#         'title': 'Paper C',
#         'publication_year': 2025,
#         'type': 'article',
#         'open_access': {'is_oa': False},
#         'primary_location': {'source': None}
#     },
#     {
#         'id': 'W4',
#         'title': 'Paper D',
#         'publication_year': 2025,
#         'type': 'article',
#         'open_access': None
#         # primary_location is completely missing
#     }
# ]
# for publication in test_papers:
#     result = paper_transform(publication)
#     print(result)

In [ ]:
# sources 
# define new function for creating the dict
def sources_transform(paper_extracted):
   
   # conditionals for source
   primary_location = paper_extracted.get('primary_location')
   if primary_location is not None:
      source = primary_location.get('source')
   else: 
      source = None   
   
   # conditional for source_id and display_name (one conditional because they are both nested under source)      
   if source is not None:
      source_id = source.get('id')
      display_name = source.get('display_name')
   else:
      source_id = None
      display_name = None
   
   source = {'id': source_id,
            'name': display_name        
         }
   return source

source = []
for journal in new_data['results']:
   source.append(sources_transform(journal))
# the code then creates a list paper that will correspond to the table, and it populates the list with a dictionary created from each element in new_data['results']


In [ ]:
# Topic
def topic_transform(paper_extracted):
    extracted_records = []
    
    # conditional for missing Topic (topics is a list, so if empty, create an empty list)
    topics = paper_extracted.get('topics')
    if topics is None:
        topics = []

    for entry in topics:
            # NEW: Handle missing field
        field = entry.get('field')
        if field is not None:
            display_name_field = field.get('display_name')
        else:
            display_name_field = None
            
        # conditional for missing value display_name (subfield)
        subfield = entry.get('subfield')
        if subfield is not None:
            display_name_subfield = subfield.get('display_name')
        else: 
            display_name_subfield = None
        #conditional missing value display_name (domain)
        domain = entry.get('domain')
        if domain is not None:
            display_name_domain = domain.get('display_name')
        else:
            display_name_domain = None
        
        record = {
            'id': entry.get('id'),
            'name': entry.get('display_name'),
            'field': display_name_field,
            'subfield': display_name_subfield,
            'domain': display_name_domain
        }
        extracted_records.append(record)

    return extracted_records

# create data for table
topic = []
for publication in new_data['results']:
    topic.extend(topic_transform(publication))
# print(len(topic)) This prints how many topics
# print(topic)

In [ ]:
# Institution 
def institution_transform (paper_extracted):
    extracted_inst = []
    # missing value authorship
    authorships = paper_extracted.get('authorships')
    if authorships is None:
        authorships = []
        
    for entry in authorships:
        institution_list = entry.get('institutions')
        # missing value institutions
        if institution_list is None:
            institution_list = []
        for inst in institution_list:
            institution = {
                'id': inst.get('id'),
                'name': inst.get('display_name')
                }
            extracted_inst.append(institution)
    return extracted_inst

# create data for table
institution = []
for i in new_data['results']:
    institution.extend(institution_transform(i))
#print(institution)

In [ ]:
# Author 
def author_transform (paper_extracted):
    author_extracted = []
    # missing value authorships 
    authorships = paper_extracted.get('authorships')
    if authorships is None:
        authorships = []
        
    for item in authorships:
        # missing value author
        author = item.get('author')
        if author is not None:
            id_author = author.get('id')
            name_author = author.get('display_name')
            orcid_author = author.get('orcid')
        else:
            id_author = None
            name_author = None 
            orcid_author = None
            
        author_dict = {
            'id': id_author,
            'name': name_author,
            'orcid': orcid_author
            }
        author_extracted.append(author_dict)
    return author_extracted 

# create data for table
author = []
for i in new_data['results']:
    author.extend(author_transform(i))


In [ ]:
# Authorship 
# define extraction function
def authorships_transform(paper_extracted):
    extracted_records = []
    # missing value authorships 
    authorships = paper_extracted.get('authorships')
    if authorships is None:
        authorships = []
    for item in authorships:
        # missing value author
        author = item.get('author')
        if author is not None:
            author_id = author.get('id')
        else:
            author_id = None
        authorships_dict = {
            'author_id': author_id,
            'paper_id': paper_extracted.get('id')             
        }
        extracted_records.append(authorships_dict)
    return extracted_records

# Create Table
authorship = []
for publication in new_data['results']:
    authorship.extend(authorships_transform(publication))

In [ ]:
# inst_author
def inst_auth_transform(paper_extracted):
    extracted_records = []
        # missing value authorships (list)
    authorships = paper_extracted.get('authorships')
    if authorships is None:
        authorships = []
    for item in authorships:
        # missing value author_id
        author = item.get('author')
        if author is not None:
            author_id = author.get('id')
        else: 
            author_id = None
        # missing institution (list)
        institutions_list = item.get('institutions')
        if institutions_list is None:
            institutions_list = []        
        for inst in institutions_list:
            record = {
                'inst_id': inst.get('id'),
                'author_id': author_id,
                'paper_id': paper_extracted.get('id')
                }
            extracted_records.append(record)
    return extracted_records

inst_auth = []
for items in new_data['results']:
    inst_auth.extend(inst_auth_transform(items))

In [ ]:
# paper_topic
# create function
def paper_topic_transform(paper_extracted):
    extracted_records = []
    # missing value topics
    topics = paper_extracted.get('topics')
    if topics is None:
        topics = []
    for item in topics:
        record_dict = {
            'paper_id': paper_extracted.get('id'),
            'topic_id': item.get('id')
        }
        extracted_records.append(record_dict)
    return extracted_records

# create table
paper_topic = []
for entry in new_data['results']:
    paper_topic.extend(paper_topic_transform(entry))


In [ ]:
# Remove duplicates from entity tables

paper = remove_duplicates(paper, ['id'])
source = remove_duplicates(source, ['id'])
topic = remove_duplicates(topic, ['id'])
institution = remove_duplicates(institution, ['id'])
author = remove_duplicates(author, ['id'])

# Remove duplicates from relationship tables

authorship = remove_duplicates(authorship, ['author_id', 'paper_id'])
paper_topic = remove_duplicates(paper_topic, ['paper_id', 'topic_id'])
inst_auth = remove_duplicates(inst_auth, ['inst_id', 'author_id', 'paper_id'])

In [ ]:
# validation 

# Validate required fields
paper = validate_records(paper, ['id', 'title', 'year'])
author = validate_records(author, ['id', 'name'])
institution = validate_records(institution, ['id'])
source = validate_records(source, ['id'])
topic = validate_records(topic, ['id', 'name', 'field'])

authorship = validate_records(authorship, ['author_id', 'paper_id'])
paper_topic = validate_records(paper_topic, ['paper_id', 'topic_id'])
inst_auth = validate_records(inst_auth, ['author_id', 'paper_id', 'inst_id'])